# Stage 3 · Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedzait20003/TableMind/blob/main/src/notebooks/03_evaluation.ipynb)

Loads the pretrained DoT model from Drive and evaluates it on the WikiSQL test split: loss and token accuracy (real answer tokens only), then exact-match accuracy of generated answers, overall and per aggregation type.

**Requires:** `02_pretrain.ipynb`

## Setup

On Colab, pick **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
# Environment setup: clones the repo on Colab, uses the local checkout otherwise
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/mohamedzait20003/TableMind.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/TableMind"):
        !git clone -q {REPO_URL} /content/TableMind
    %cd /content/TableMind
    !pip install -q -r requirements.txt
else:
    # Local run (e.g. from src/notebooks): move up to the repository root
    os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()))

sys.path.insert(0, os.path.abspath("src"))
print("Working directory:", os.getcwd())

In [ ]:
from utils import load_config, set_seed, resolve_device, get_storage_dir

# quick.yaml: t5-small + tapas-small, small subsets (fits a free T4)
# default.yaml: t5-base + tapas-large, the full experiment
CONFIG_PATH = "src/config/quick.yaml"

config = load_config(CONFIG_PATH)
device = resolve_device(config["system"]["device"])
set_seed(config["system"]["seed"])

# Google Drive on Colab (you will be asked to authorize), ./storage locally
STORAGE = get_storage_dir(config["storage"]["drive_subdir"], config["storage"]["local_dir"])
CHECKPOINT_DIR = config["system"]["checkpoint_dir"]
OUTPUT_DIR = config["system"]["output_dir"]
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Config: {CONFIG_PATH} ({config['experiment']['name']})")
print(f"Device: {device} | storage: {STORAGE}")

## Load the pretrained model

In [ ]:
from core import MemoryEncoder, KeyValueMemoryStore, DoTModel
from utils import download_checkpoint, load_checkpoint

enc_cfg = config["model"]["memory_encoder"]
memory_encoder = MemoryEncoder(model_name=enc_cfg["model_name"], proj_dim=enc_cfg["proj_dim"])
memory_store = KeyValueMemoryStore()
model = DoTModel(
    memory_store, memory_encoder,
    k=config["model"]["pruning_transformer"]["top_k"],
    pruning_model=config["model"]["pruning_transformer"]["model_name"],
    task_model=config["model"]["task_transformer"]["model_name"],
    memory_top_k=config["pretraining"]["memory_top_k"],
    task_max_length=config["model"]["task_transformer"]["max_length"],
)

model_path = download_checkpoint(config["storage"]["pretrain_checkpoint"], STORAGE, CHECKPOINT_DIR)
checkpoint = load_checkpoint(model_path, model, memory_store=memory_store)
model.to(device).eval()
print("Memory entries:", memory_store.size())

## Load the WikiSQL test split

In [ ]:
from torch.utils.data import DataLoader
from utils import WikiSQLDataset, collate_fn

test_dataset = WikiSQLDataset.from_config(config, "test")
test_loader = DataLoader(test_dataset, batch_size=config["pretraining"]["batch_size"],
                         collate_fn=collate_fn, num_workers=config["system"]["num_workers"])
print("Test examples:", len(test_dataset))

## Loss and token accuracy

In [ ]:
from core import PretrainTrainer

evaluator = PretrainTrainer(model, None, test_loader, device=device)
metrics = evaluator.evaluate()
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

## Exact match

Predictions are compared to the executed SQL answers, ignoring case and whitespace. Set `MAX_BATCHES = None` to score the whole test subset.

In [ ]:
import pandas as pd
from utils import exact_match

MAX_BATCHES = 50

em, records = exact_match(
    model, test_loader, device,
    max_new_tokens=config["data"]["wikisql"]["answer_max_length"],
    max_batches=MAX_BATCHES,
)
print(f"Exact match: {em:.4f} on {len(records)} examples")

results = pd.DataFrame(records)
# test_loader is not shuffled, so record i is test example i
results["question"] = [test_dataset.data[i].question for i in range(len(results))]
results["aggregation"] = [test_dataset.data[i].aggregation.name for i in range(len(results))]
normalize = lambda s: "".join(s.lower().split())
results["correct"] = results["prediction"].map(normalize) == results["answer"].map(normalize)
results[["question", "answer", "prediction", "correct"]].head(20)

In [ ]:
results.groupby("aggregation")["correct"].agg(exact_match="mean", examples="count")

## Inspect pruning on one example

What T5 actually sees after TAPAS keeps the top-k tokens, with each table word's `sigmoid(score)` scale.

In [ ]:
import torch

batch = next(iter(test_loader))
with torch.no_grad():
    t5_ids, t5_mask, scale = model.task_inputs(
        batch["input_ids"][:1].to(device),
        batch["attention_mask"][:1].to(device),
        batch["token_type_ids"][:1].to(device),
    )

tokenizer = model.task_model.tokenizer
print("Question:", test_dataset.data[0].question)
print("Answer:  ", test_dataset.data[0].answer)
print("\nT5 input:", tokenizer.decode(t5_ids[0], skip_special_tokens=True))

tokens = tokenizer.convert_ids_to_tokens(t5_ids[0][t5_mask[0].bool()].tolist())
weights = scale[0][t5_mask[0].bool()].tolist()
pd.DataFrame({"token": tokens, "scale": weights}).query("scale < 1").sort_values("scale", ascending=False).head(15)